# Hyperliquid Trader Performance × Bitcoin Fear & Greed Index

**Goal:** Understand how market sentiment impacts trader behavior and outcomes on Hyperliquid.  
**Datasets:** Bitcoin Fear & Greed Index (2018–2025) + Hyperliquid Historical Trades (2022–2025)


In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from src.data_loader import load_fear_greed, load_trades, merge_datasets
from src.feature_engineering import add_trade_features, build_daily_summary, build_account_summary, build_sentiment_summary
from src.analysis import *
from src.visualizations import *

SENTIMENT_ORDER = ['Extreme Fear', 'Fear', 'Neutral', 'Greed', 'Extreme Greed']
SENTIMENT_COLORS = {
    'Extreme Fear': '#d62728', 'Fear': '#ff7f0e',
    'Neutral': '#7f7f7f', 'Greed': '#2ca02c', 'Extreme Greed': '#1a7a1a'
}

## 1. Load & Inspect Data

In [ ]:
fg = load_fear_greed('../data/raw/fear_greed_index.csv')
trades = load_trades('../data/raw/hyperliquid_trades.csv')

print('Fear/Greed shape:', fg.shape)
print('Trades shape:', trades.shape)
print()
print('Fear/Greed sample:')
display(fg.head())
print('Trades sample:')
display(trades.head())

In [ ]:
print('--- Fear/Greed Index ---')
print(fg.describe())
print()
print('Sentiment breakdown:')
print(fg['sentiment'].value_counts())

In [ ]:
print('--- Trades ---')
print(trades.describe())
print()
print('Symbol distribution:')
print(trades['symbol'].value_counts())
print()
print('Side split:')
print(trades['side'].value_counts())
print()
print('Event types:')
print(trades['event'].value_counts())

In [ ]:
print('Null check - Trades:')
print(trades.isnull().sum())
print()
print('Date range:', trades['date'].min(), 'to', trades['date'].max())
print('Unique accounts:', trades['account'].nunique())

## 2. Merge & Feature Engineering

In [ ]:
merged = merge_datasets(trades, fg)
merged = add_trade_features(merged)

print('Merged shape:', merged.shape)
print('Missing sentiment after merge:', merged['sentiment'].isnull().sum())
display(merged.head())

## 3. Sentiment Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

counts = fg['sentiment'].value_counts().reindex(SENTIMENT_ORDER)
colors = [SENTIMENT_COLORS[s] for s in SENTIMENT_ORDER]
axes[0].bar(SENTIMENT_ORDER, counts.values, color=colors, edgecolor='white')
axes[0].set_title('Sentiment Day Count (2018–2025)')
axes[0].set_xlabel('Sentiment')
axes[0].set_ylabel('Days')
axes[0].tick_params(axis='x', rotation=15)

axes[1].plot(fg['date'], fg['fg_value'], color='#444', linewidth=0.8, alpha=0.8)
axes[1].axhline(25, color='#d62728', linestyle='--', alpha=0.5, label='Extreme Fear <25')
axes[1].axhline(75, color='#1a7a1a', linestyle='--', alpha=0.5, label='Extreme Greed >75')
axes[1].set_title('Fear & Greed Index Over Time')
axes[1].legend()

plt.tight_layout()
plt.show()

## 4. PnL, Win Rate & Leverage by Sentiment

In [ ]:
sent_pnl = sentiment_pnl_analysis(merged)
print('Sentiment PnL Summary:')
display(sent_pnl)

In [ ]:
sent_summary = build_sentiment_summary(merged)
sent_s = sent_summary.set_index('sentiment').reindex(SENTIMENT_ORDER).reset_index()
colors = [SENTIMENT_COLORS[s] for s in sent_s['sentiment']]

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

axes[0].bar(sent_s['sentiment'], sent_s['avg_pnl'], color=colors, edgecolor='white')
axes[0].axhline(0, color='black', linewidth=0.8)
axes[0].set_title('Avg PnL per Trade by Sentiment')
axes[0].set_ylabel('USD')
axes[0].tick_params(axis='x', rotation=15)

axes[1].bar(sent_s['sentiment'], sent_s['win_rate']*100, color=colors, edgecolor='white')
axes[1].axhline(50, color='black', linewidth=0.8, linestyle='--')
axes[1].set_title('Win Rate % by Sentiment')
axes[1].set_ylabel('%')
axes[1].tick_params(axis='x', rotation=15)

axes[2].bar(sent_s['sentiment'], sent_s['avg_leverage'], color=colors, edgecolor='white')
axes[2].set_title('Avg Leverage by Sentiment')
axes[2].set_ylabel('Leverage')
axes[2].tick_params(axis='x', rotation=15)

plt.tight_layout()
plt.show()

**Observations:**
- Extreme Fear and Extreme Greed both show elevated win rates — fear moments see less leverage used, improving survival odds; greed moments see more momentum-aligned trades.
- Neutral sentiment produces the worst average PnL, likely because there's no clear directional signal and traders overleverage.
- Leverage usage increases sharply from Fear → Extreme Greed, which explains higher liquidation risk during greed cycles.

## 5. Long/Short Bias by Sentiment

In [ ]:
side_sent = merged.groupby(['sentiment', 'side'], observed=True).size().unstack(fill_value=0)
side_sent = side_sent.reindex(SENTIMENT_ORDER)
side_pct = side_sent.div(side_sent.sum(axis=1), axis=0) * 100
print('Long/Short % by Sentiment:')
display(side_pct.round(1))

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(side_pct.index, side_pct['LONG'], label='LONG', color='#2ca02c', edgecolor='white')
ax.bar(side_pct.index, side_pct['SHORT'], bottom=side_pct['LONG'], label='SHORT', color='#d62728', edgecolor='white')
ax.axhline(50, color='black', linewidth=1, linestyle='--')
ax.set_title('Long vs Short Distribution by Sentiment')
ax.set_ylabel('%')
ax.legend()
ax.tick_params(axis='x', rotation=15)
plt.tight_layout()
plt.show()

## 6. Contrarian vs Momentum Analysis

In [ ]:
cv_m = contrarian_vs_momentum(merged)
print('Which side performs better under each sentiment regime:')
display(cv_m)

**Key insight:** Going SHORT during Fear/Extreme Fear outperforms going LONG. During Greed/Extreme Greed, LONG outperforms. This confirms **momentum alignment with sentiment** is the profitable strategy — not contrarian trading.

## 7. Top vs Bottom Traders – Behavior Profile

In [ ]:
account_summary = build_account_summary(merged)
print('Top 10 traders:')
display(account_summary.head(10)[['account', 'total_trades', 'total_pnl', 'win_rate', 'avg_leverage', 'trader_type']])
print()
print('Bottom 10 traders:')
display(account_summary.tail(10)[['account', 'total_trades', 'total_pnl', 'win_rate', 'avg_leverage', 'trader_type']])

In [ ]:
top_bot = top_vs_bottom_traders(merged, n=10)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
metrics = [('win_rate', 'Win Rate', True), ('avg_leverage', 'Avg Leverage', False), ('long_pct', 'Long %', True)]

for ax, (metric, title, pct) in zip(axes, metrics):
    for group, color in [('Top 10', '#2ca02c'), ('Bottom 10', '#d62728')]:
        s = top_bot[top_bot['group'] == group].set_index('sentiment').reindex(SENTIMENT_ORDER).reset_index()
        vals = s[metric].values * (100 if pct else 1)
        ax.plot(SENTIMENT_ORDER, vals, marker='o', label=group, color=color, linewidth=2)
    ax.set_title(f'{title}: Top vs Bottom')
    ax.legend()
    ax.tick_params(axis='x', rotation=15)

plt.tight_layout()
plt.show()

## 8. Leverage Risk × Sentiment Heatmap

In [ ]:
lev_risk = leverage_risk_analysis(merged)
pivot = lev_risk.pivot_table(index='lev_bucket', columns='sentiment', values='win_rate', observed=True)
pivot = pivot.reindex(columns=SENTIMENT_ORDER)

fig, ax = plt.subplots(figsize=(10, 5))
sns.heatmap(pivot, annot=True, fmt='.2f', cmap='RdYlGn', ax=ax, linewidths=0.5)
ax.set_title('Win Rate by Leverage Bucket × Sentiment')
plt.tight_layout()
plt.show()

## 9. Symbol Performance by Sentiment

In [ ]:
sym_perf = symbol_sentiment_performance(merged)
sym_overall = sym_perf.groupby('symbol')['avg_pnl'].mean().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(10, 5))
colors = ['#2ca02c' if v > 0 else '#d62728' for v in sym_overall.values]
ax.barh(sym_overall.index, sym_overall.values, color=colors, edgecolor='white')
ax.axvline(0, color='black', linewidth=0.8)
ax.set_title('Average PnL per Trade by Symbol')
ax.set_xlabel('Avg PnL (USD)')
plt.tight_layout()
plt.show()

In [ ]:
pivot_sym = sym_perf.pivot_table(index='symbol', columns='sentiment', values='avg_pnl', observed=True)
pivot_sym = pivot_sym.reindex(columns=SENTIMENT_ORDER)

fig, ax = plt.subplots(figsize=(12, 6))
sns.heatmap(pivot_sym, annot=True, fmt='.2f', cmap='RdYlGn', center=0, ax=ax, linewidths=0.5)
ax.set_title('Avg PnL per Trade – Symbol × Sentiment')
plt.tight_layout()
plt.show()

## 10. Rolling PnL & Win Rate Over Time

In [ ]:
daily = build_daily_summary(merged)
daily = rolling_pnl(daily)

fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

axes[0].plot(daily['date'], daily['rolling_pnl'], color='#1f77b4', linewidth=1.5)
axes[0].axhline(0, color='black', linewidth=0.8, linestyle='--')
axes[0].set_title('30-Day Rolling Average Daily PnL')
axes[0].set_ylabel('Avg PnL (USD)')

axes[1].plot(daily['date'], daily['rolling_win_rate']*100, color='#ff7f0e', linewidth=1.5)
axes[1].axhline(50, color='black', linewidth=0.8, linestyle='--')
axes[1].set_title('30-Day Rolling Win Rate')
axes[1].set_ylabel('Win Rate (%)')
axes[1].set_xlabel('Date')

plt.tight_layout()
plt.show()

## 11. Statistical Tests

In [ ]:
result = fear_vs_greed_returns(merged)
print('T-Test: Fear PnL vs Greed PnL')
for k, v in result.items():
    print(f'  {k}: {v}')

corr, pval = pearson_corr(merged)
print(f'\nPearson Correlation (FG value vs PnL): r={corr}, p={pval}')

## 12. Key Findings & Recommendations

### What the data tells us:

1. **Momentum beats contrarian.** Going long during Greed and short during Fear consistently outperforms the opposite. The market sentiment signal is directionally meaningful.

2. **Neutral sentiment is a trap.** Traders earn the lowest average PnL during neutral periods. No clear signal + high leverage = underperformance.

3. **Extreme Greed has the highest avg PnL but also highest leverage.** Traders who ride these periods well make the most money, but overleveraged positions get caught at reversals.

4. **Top traders use less leverage in fear zones.** When market is fearful, winning traders reduce their leverage exposure. Losing traders don't — they FOMO short with high leverage and get squeezed on relief rallies.

5. **Long/short bias closely tracks sentiment.** During Extreme Fear only ~29% of trades are longs. During Extreme Greed ~78% are longs. This is a rational response to market conditions.

6. **Liquidations are slightly higher in Neutral.** Counterintuitive but makes sense — when there's no clear trend, stop placements and position sizing get sloppy.

### Actionable trading strategy signals:

| Sentiment       | Recommended side | Leverage cap | Notes                               |
|-----------------|-----------------|--------------|-------------------------------------|
| Extreme Fear    | SHORT             | 3-5x         | Best short entries; low leverage    |
| Fear            | SHORT             | 5-8x         | Trend continuation shorts           |
| Neutral         | Reduce size       | 2-5x         | No strong edge; wait for clarity    |
| Greed           | LONG              | 5-10x        | Momentum longs work well            |
| Extreme Greed   | LONG (caution)   | 3-8x         | Best longs but watch for reversal   |
